# Predicting whether a Netflix title is rated 18+ from its description
Multinomial Naive Bayes on the Kaggle Netflix titles dataset.

## 1. Imports and load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score, f1_score

df = pd.read_csv("netflix_titles.csv")
print(df.shape)
df.head()

## 2. Data quality checks (before cleaning)

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate descriptions:", df["description"].duplicated().sum())

In [ ]:
print(df["rating"].value_counts(dropna=False))

## 3. Cleaning
- Keep only valid age ratings (drops missing ratings, NR/UR unrated titles, and the 3 rows with durations like "74 min" in the rating column)
- Drop missing and duplicate descriptions
- Create the target: **18+** = R, TV-MA, NC-17; everything else = **Not 18+**

In [ ]:
valid = ["TV-Y", "TV-Y7", "TV-Y7-FV", "TV-G", "G", "PG", "TV-PG",
         "PG-13", "TV-14", "R", "TV-MA", "NC-17"]

before = len(df)
df = df[df["rating"].isin(valid)]
df = df.dropna(subset=["description"])
df = df.drop_duplicates(subset="description")
print(f"Rows: {before} -> {len(df)} ({before - len(df)} removed)")

df["adult"] = df["rating"].isin(["R", "TV-MA", "NC-17"]).map({True: "18+", False: "Not 18+"})
print(df["adult"].value_counts())

## 4. Exploratory data analysis

### 4.1 Target distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["rating"].value_counts().plot(kind="bar", ax=axes[0], title="Titles per rating")
df["adult"].value_counts().plot(kind="bar", ax=axes[1], title="18+ vs Not 18+")
plt.tight_layout()
plt.show()

print("Baseline accuracy (always guess the majority class):",
      round(df["adult"].value_counts(normalize=True).max(), 3))

### 4.2 Description length

In [ ]:
df["desc_len"] = df["description"].str.split().str.len()
print(df["desc_len"].describe())
print(df.groupby("adult")["desc_len"].mean().round(1))

sns.boxplot(data=df, x="adult", y="desc_len")
plt.title("Description length (words) by group")
plt.show()

### 4.3 Most common words per group

In [ ]:
for g in ["18+", "Not 18+"]:
    cv = CountVectorizer(stop_words="english")
    counts = cv.fit_transform(df.loc[df["adult"] == g, "description"]).sum(axis=0).A1
    top = pd.Series(counts, index=cv.get_feature_names_out()).nlargest(15)
    print(g, list(top.index))

### 4.4 Movie vs TV Show

In [ ]:
pd.crosstab(df["type"], df["adult"], normalize="index").plot(kind="bar", stacked=True)
plt.title("Share of 18+ titles: Movie vs TV Show")
plt.ylabel("Proportion")
plt.show()

## 5. Train/test split
Stratified so both sets keep the same 18+ / Not 18+ proportions. The vectorizer is inside the pipeline, so it only learns vocabulary from the training set.

In [ ]:
X = df["description"]
y = df["adult"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train:", len(X_train), " Test:", len(X_test))

## 6. Multinomial Naive Bayes
Word counts (CountVectorizer) + Multinomial NB. Alpha (smoothing) is tuned with 5-fold cross-validation on the training set only.

In [ ]:
model = make_pipeline(
    CountVectorizer(stop_words="english"),
    MultinomialNB(),
)

grid = GridSearchCV(
    model,
    {"multinomialnb__alpha": [0.1, 0.5, 1, 2]},
    scoring="f1_macro",
    cv=5,
)
grid.fit(X_train, y_train)

print("Best alpha:", grid.best_params_)
print("Best cross-validation macro-F1:", round(grid.best_score_, 3))

## 7. Evaluation on the test set

In [ ]:
y_pred = grid.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Macro-F1:", round(f1_score(y_test, y_pred, average="macro"), 3))
print()
print(classification_report(y_test, y_pred))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=["18+", "Not 18+"], cmap="Blues")
plt.title("Confusion matrix")
plt.show()

## 8. What the model learned
Words whose probability is much higher in one group than the other.

In [ ]:
nb = grid.best_estimator_.named_steps["multinomialnb"]
words = np.array(grid.best_estimator_.named_steps["countvectorizer"].get_feature_names_out())

diff = nb.feature_log_prob_[0] - nb.feature_log_prob_[1]   # class 0 minus class 1
print(nb.classes_[0], "words:", list(words[np.argsort(diff)[-15:]][::-1]))
print(nb.classes_[1], "words:", list(words[np.argsort(diff)[:15]]))

## 9. Error analysis
Which original ratings get misclassified most often?

In [ ]:
results = df.loc[X_test.index, ["title", "rating", "description"]].copy()
results["true"] = y_test
results["pred"] = y_pred
results["correct"] = results["true"] == results["pred"]

print(results.groupby("rating")["correct"].mean().sort_values().round(2))

In [ ]:
# A few misclassified examples
results[~results["correct"]][["title", "rating", "pred", "description"]].head(10)